# EHR Adoption — Validate Dimensional Model
Run after `2_etl.py` to verify row counts and run analytical queries.

In [ ]:
import snowflake.connector
import configparser
import pandas as pd

config = configparser.ConfigParser()
config.read('../snowflake.cfg')

conn = snowflake.connector.connect(
    user=config['SNOWFLAKE']['user'],
    password=config['SNOWFLAKE']['password'],
    account=config['SNOWFLAKE']['account'],
    warehouse=config['SNOWFLAKE']['warehouse'],
    database=config['SNOWFLAKE']['database'],
    schema=config['SNOWFLAKE']['schema'],
)

def q(sql):
    return pd.read_sql(sql, conn)

## 1. Row Counts

In [ ]:
q("""
SELECT 'fact_ehr_adoptions' AS tbl, COUNT(*) AS rows FROM fact_ehr_adoptions
UNION ALL SELECT 'fact_workflow_metrics', COUNT(*) FROM fact_workflow_metrics
UNION ALL SELECT 'dim_providers', COUNT(*) FROM dim_providers
UNION ALL SELECT 'dim_program_year', COUNT(*) FROM dim_program_year
UNION ALL SELECT 'dim_ehr_vendor', COUNT(*) FROM dim_ehr_vendor
""")

## 2. Adoption Score by Region and Year

In [ ]:
q("""
SELECT p.region, y.program_year,
       ROUND(AVG(f.adoption_score), 2) AS avg_adoption_score
FROM fact_ehr_adoptions f
JOIN dim_providers p ON f.provider_key = p.provider_key
JOIN dim_program_year y ON f.year_key = y.year_key
GROUP BY p.region, y.program_year
ORDER BY p.region, y.program_year
""")

## 3. Telehealth Adoption Pre vs Post COVID

In [ ]:
q("""
SELECT y.program_year, y.era,
       ROUND(AVG(f.telehealth_enabled) * 100, 1) AS telehealth_pct
FROM fact_ehr_adoptions f
JOIN dim_program_year y ON f.year_key = y.year_key
GROUP BY y.program_year, y.era
ORDER BY y.program_year
""")

## 4. Workflow Quality by Certification Level

In [ ]:
q("""
SELECT f.certification_level,
       ROUND(AVG(m.avg_documentation_time_min), 1) AS avg_doc_time,
       ROUND(AVG(m.data_completeness_pct), 1) AS avg_data_completeness,
       ROUND(AVG(m.patient_satisfaction_score), 2) AS avg_satisfaction
FROM fact_workflow_metrics m
JOIN fact_ehr_adoptions f ON m.adoption_key = f.adoption_key
GROUP BY f.certification_level
ORDER BY avg_doc_time
""")